<a href="https://colab.research.google.com/github/CefasRepRes/shipwreck-oil-detection-colab/blob/main/CefasCobblesTools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def install_requirements():
  from google.colab import output
  output.enable_custom_widget_manager()
  %matplotlib ipympl
  import torch
  import torchvision
  print("PyTorch version:", torch.__version__)
  print("Torchvision version:", torchvision.__version__)
  print("CUDA is available:", torch.cuda.is_available())
  import sys
  !{sys.executable} -m pip install opencv-python matplotlib
  !{sys.executable} -m pip install sam2
  !{sys.executable} -m pip install segmenteverygrain
  !{sys.executable} -m pip install rtree
  !wget https://huggingface.co/facebook/sam2.1-hiera-large/resolve/main/sam2.1_hiera_large.pt
  from sam2.build_sam import build_sam2
  from sam2.sam2_image_predictor import SAM2ImagePredictor
  sam_checkpoint = "sam2.1_hiera_large.pt"
  device = "cuda"
  sam = build_sam2("configs/sam2.1/sam2.1_hiera_l.yaml", sam_checkpoint, device=device)
  predictor = SAM2ImagePredictor(sam)
  import pandas as pd
  import numpy as np
  import matplotlib.pyplot as plt
  import cv2
  from skimage import measure
  from importlib import reload
  import segmenteverygrain as seg
  import segmenteverygrain.interactions as si
  from tqdm import trange, tqdm
  import pickle
  from pathlib import Path
  !wget "https://raw.githubusercontent.com/zsylvester/segmenteverygrain/main/models/seg_model_smooth_labels.pt"
  # U-Net model (PyTorch)
  unet = seg.load_unet("seg_model_smooth_labels.pt")
  !wget -O WWBF_CEND0121_WWBF008_STN_182_A1_006.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF008_STN_182_A1_006.JPG
  !wget -O WWBF_CEND0121_WWBF008_STN_182_A1_011.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF008_STN_182_A1_011.JPG
  !wget -O WWBF_CEND0121_WWBF008_STN_182_A1_020.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF008_STN_182_A1_020.JPG
  !wget -O WWBF_CEND0121_WWBF013_STN_180_A1_005.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF013_STN_180_A1_005.JPG
  !wget -O WWBF_CEND0121_WWBF013_STN_180_A1_023.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF013_STN_180_A1_023.JPG
  !wget -O WWBF_CEND0121_WWBF019_STN_174_A1_022.JPG https://raw.githubusercontent.com/CefasRepRes/shipwreck-oil-detection-colab/main/WWBF_CEND0121_WWBF019_STN_174_A1_022.JPG


Code to create the image dropdown box

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
def Select_image():
  # Folder containing the images
  image_dir = Path("/content")

  # Find JPG files, ignoring case
  image_files = sorted([
     f for f in image_dir.iterdir()
      if f.is_file() and f.suffix.lower() == ".jpg"
  ])

  # Dropdown shows filename without extension
  IMGdropdown = widgets.Dropdown(
     options=[f.stem for f in image_files],
     description="Select image:"
  )
  # Slider for pixel filter
  min_area_px = widgets.IntSlider(value=250, min=50, max=750, step=1, description='Pixel Filter', readout=True, orientation='horizontal', layout=widgets.Layout(width='400px'), style={'Min pixel area': 'auto'})
  dbs__max_dist = widgets.IntSlider(value=30, min=10, max=500, step=1, description='Pixel Filter', readout=True, orientation='horizontal', layout=widgets.Layout(width='400px'), style={'Prompt distance': 'auto'})

  return display(IMGdropdown, min_area_px,dbs__max_dist)


Dropdown(description='Image:', options=('WWBF_CEND0121_WWBF008_STN_182_A1_006', 'WWBF_CEND0121_WWBF008_STN_182…